# GMSC Credit Risk: Scoring, Scenarios & Recommendations Engine
1. **Scoring + what-if scenarios** (Stage 10)
2. **Recommendations Engine** (Stage 11): the smallest change the customer can make to lower their risk.
3. **Savings Coverage** (Stage 12): rule-based runway / shock coverage shown next to every scenario.
4. **Long-Term Risk View, 24 months** (Stage 13): structural GMSC risk with plain-language drivers.

Requires `gmsc_final_model.joblib` in the working directory.


In [ ]:
# pip install joblib shap xgboost scikit-learn pandas numpy
# On Colab, mount Drive and point to the folder that contains gmsc_final_model.joblib:
# from google.colab import drive; drive.mount("/content/drive")
# import os; os.chdir("/content/drive/MyDrive/<your-folder>")
import os
print("Working dir:", os.getcwd())
print("Model found:", os.path.exists("gmsc_final_model.joblib"))


## 1. Scoring + What-if scenarios

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import shap

# --------------------------------------------------------------------------
# Model loading (path is relative to this file, so it works anywhere)
# --------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
MODEL_PATH = os.path.join(BASE_DIR, "gmsc_final_model.joblib")
ART = joblib.load(MODEL_PATH)

PIPE, CAL, THRESHOLD = ART["xgb_pipeline"], ART["calibrator"], ART["threshold"]
FEATURE_COLS = ART["features"]
LOWER, UPPER = ART["clip_lower"], ART["clip_upper"]
NUMERIC_COLS = list(LOWER.index)

RAW_COLS = [
    "RevolvingUtilizationOfUnsecuredLines", "age", "NumberOfTime30-59DaysPastDueNotWorse", "DebtRatio",
    "MonthlyIncome", "NumberOfOpenCreditLinesAndLoans", "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfDependents",
]


# --------------------------------------------------------------------------
# Feature engineering
# --------------------------------------------------------------------------
def build_gmsc_features_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # duplicate review
    id_like_cols = [c for c in df.columns if c.startswith("Unnamed")]
    dup_check_cols = [c for c in df.columns if c not in id_like_cols]
    duplicate_flag = df.duplicated(subset=dup_check_cols, keep=False).astype(int)
    df = df.drop(columns=id_like_cols)

    out = pd.DataFrame(index=df.index)
    out["duplicate_flag"] = duplicate_flag.values

    out["income_missing_flag"] = df["MonthlyIncome"].isna().astype(int)
    out["income"] = df["MonthlyIncome"]

    valid_income_mask = out["income"].notna() & (out["income"] > 0)
    debt_ratio_clean = df["DebtRatio"].where(valid_income_mask)
    out["derived_debt_ratio_outlier"] = (df["DebtRatio"] > 5).astype(int)

    past_due_cols = [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
    ]
    placeholder_mask = pd.Series(False, index=df.index)
    past_due_clean = pd.DataFrame(index=df.index)
    for col in past_due_cols:
        is_placeholder = df[col].isin([96, 98])
        placeholder_mask = placeholder_mask | is_placeholder
        past_due_clean[col] = df[col].where(~is_placeholder)
    out["derived_placeholder_code_flag"] = placeholder_mask.astype(int)

    out["age"] = df["age"].replace(0, np.nan)

    out["dependents_missing_flag"] = df["NumberOfDependents"].isna().astype(int)
    out["dependents"] = df["NumberOfDependents"].fillna(0)

    out["derived_late_count"] = past_due_clean.sum(axis=1, skipna=True)
    out["derived_late_severity"] = (
        past_due_clean["NumberOfTime30-59DaysPastDueNotWorse"].fillna(0) * 1
        + past_due_clean["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0) * 2
        + past_due_clean["NumberOfTimes90DaysLate"].fillna(0) * 3
    )
    out["late_payments"] = out["derived_late_count"]

    obligations_mask = valid_income_mask & (df["DebtRatio"] <= 5)
    out["derived_obligations"] = np.where(obligations_mask, df["DebtRatio"] * out["income"], np.nan)
    out["monthly_debt_payment"] = out["derived_obligations"]
    out["monthly_debt_payment_is_proxy"] = 1

    out["derived_expense_ratio"] = debt_ratio_clean
    out["derived_monthly_surplus"] = out["income"] - out["derived_obligations"]
    out["savings"] = out["derived_monthly_surplus"]
    out["savings_is_proxy"] = 1

    out["derived_overspend_flag"] = (debt_ratio_clean > 1).astype(int)

    util = df["RevolvingUtilizationOfUnsecuredLines"]
    out["derived_utilization_capped"] = util.clip(upper=1)
    out["derived_utilization_outlier_flag"] = (util > 1).astype(int)
    out["revolving_utilization_ref"] = util

    out["derived_log_income"] = np.log1p(out["income"].clip(lower=0))

    for c in ["debt", "expenses", "rent", "income_stability", "month"]:
        out[c] = np.nan

    out["target"] = df["SeriousDlqin2yrs"]
    out["source_dataset"] = "gmsc"
    return out


# --------------------------------------------------------------------------
# Scoring + explanations
# --------------------------------------------------------------------------
pre, model = PIPE.named_steps["pre"], PIPE.named_steps["clf"]
RAW_NAMES = [n.split("__", 1)[-1] for n in pre.get_feature_names_out()]
explainer = shap.TreeExplainer(model)

GROUPS = {
    "Late payments":      ["derived_late_count", "derived_late_severity", "derived_placeholder_code_flag"],
    "Credit utilization": ["derived_utilization_capped", "derived_utilization_outlier_flag"],
    "Income":             ["derived_log_income", "derived_obligations", "derived_monthly_surplus", "income_missing_flag"],
    "Debt ratio":         ["derived_expense_ratio", "derived_debt_ratio_outlier", "derived_overspend_flag"],
    "Age":                ["age"],
    "Dependents":         ["dependents", "dependents_missing_flag"],
}
assert sorted(sum(GROUPS.values(), [])) == sorted(RAW_NAMES)

MEDIUM_CUT = THRESHOLD / 2  # design decision: agree on this with the team


def risk_level(p):
    return "High" if p >= THRESHOLD else "Medium" if p >= MEDIUM_CUT else "Low"


def _as_frame(customer):
    df = customer.to_frame().T if isinstance(customer, pd.Series) else \
         customer.copy() if isinstance(customer, pd.DataFrame) else pd.DataFrame([customer])
    missing = [c for c in RAW_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing raw columns: {missing}")
    return df[RAW_COLS].astype(float).reset_index(drop=True)


def _prepare(raw_df):
    raw_df = raw_df.copy()
    raw_df["SeriousDlqin2yrs"] = np.nan
    X = build_gmsc_features_df(raw_df)[FEATURE_COLS].copy()
    X[NUMERIC_COLS] = X[NUMERIC_COLS].clip(lower=LOWER, upper=UPPER, axis=1)  # training bounds
    return X


def _reasons(X, top_k):
    s = pd.Series(explainer.shap_values(pre.transform(X))[0], index=RAW_NAMES)
    g = pd.Series({k: s[v].sum() for k, v in GROUPS.items()})
    g = g.sort_values(key=np.abs, ascending=False).head(top_k)
    return [{"driver": k, "effect": "increases risk" if v > 0 else "decreases risk",
             "shap": round(float(v), 3)} for k, v in g.items()]


def score_customer(customer, top_k=3):
    X = _prepare(_as_frame(customer))
    p = float(CAL.predict(PIPE.predict_proba(X)[:, 1])[0])
    return {
        "dataset": "gmsc", "horizon": "24 months",
        "risk": round(p, 4), "level": risk_level(p),
        "flagged": p >= THRESHOLD, "threshold": round(THRESHOLD, 4),
        "top_reasons": _reasons(X, top_k),
    }


# --------------------------------------------------------------------------
# What-if scenarios
# --------------------------------------------------------------------------
def _need_income(c):
    if not (pd.notna(c["MonthlyIncome"]) and c["MonthlyIncome"] > 0 and pd.notna(c["DebtRatio"])):
        raise ValueError("This scenario needs a valid MonthlyIncome and DebtRatio")


def income_drop(c, pct):
    c = c.copy(); _need_income(c)
    obligations = c["DebtRatio"] * c["MonthlyIncome"]       # actual amount stays fixed
    c["MonthlyIncome"] *= (1 - pct)
    c["DebtRatio"] = obligations / c["MonthlyIncome"]       # the ratio is what goes up
    return c


def add_monthly_payment(c, amount):                         # new loan / higher bill
    c = c.copy(); _need_income(c)
    c["DebtRatio"] += amount / c["MonthlyIncome"]
    return c


def late_payment(c, months=1):                              # a late payment
    c = c.copy()
    col = "NumberOfTime30-59DaysPastDueNotWorse"
    c[col] = (0 if pd.isna(c[col]) else c[col]) + months
    return c


SCENARIOS = {
    "Income drops 20%":       lambda c: income_drop(c, 0.20),
    "New loan (1,500/mo)":    lambda c: add_monthly_payment(c, 1500),
    "Higher bills (+800/mo)": lambda c: add_monthly_payment(c, 800),
    "One month late":         lambda c: late_payment(c, 1),
}


def run_scenario(customer, name, change_fn=None):
    c = _as_frame(customer).iloc[0]
    before = score_customer(c)
    after = score_customer(change_fn(c) if change_fn else SCENARIOS[name](c))
    return {"scenario": name, "before": before["risk"], "after": after["risk"],
            "delta": round(after["risk"] - before["risk"], 4),
            "level_before": before["level"], "level_after": after["level"],
            "reasons_after": after["top_reasons"]}


def run_all(customer):
    out = []
    for name in SCENARIOS:
        try:
            out.append(run_scenario(customer, name))
        except ValueError as e:
            out.append({"scenario": name, "error": str(e)})
    return pd.DataFrame(out)


# --------------------------------------------------------------------------
# Friendly input -> raw GMSC columns
# --------------------------------------------------------------------------
def build_raw_customer(age, monthly_income, monthly_debt_payments, card_balance, card_limit,
                       late_30_59=0, late_60_89=0, late_90_plus=0, dependents=0,
                       open_credit_lines=0, real_estate_loans=0):
    if not 18 <= age <= 100:               raise ValueError("Age must be between 18 and 100")
    if monthly_income <= 0:                raise ValueError("Monthly income must be greater than 0")
    if monthly_debt_payments < 0:          raise ValueError("Debt payments can't be negative")
    if card_limit < 0 or card_balance < 0: raise ValueError("Card balance/limit can't be negative")
    if min(late_30_59, late_60_89, late_90_plus, dependents) < 0:
        raise ValueError("Counts can't be negative")

    utilization = card_balance / card_limit if card_limit > 0 else 0.0
    return {
        "RevolvingUtilizationOfUnsecuredLines": utilization,
        "age": age,
        "NumberOfTime30-59DaysPastDueNotWorse": late_30_59,
        "DebtRatio": monthly_debt_payments / monthly_income,
        "MonthlyIncome": monthly_income,
        "NumberOfOpenCreditLinesAndLoans": open_credit_lines,
        "NumberOfTimes90DaysLate": late_90_plus,
        "NumberRealEstateLoansOrLines": real_estate_loans,
        "NumberOfTime60-89DaysPastDueNotWorse": late_60_89,
        "NumberOfDependents": dependents,
    }


## 2. Recommendations Engine (Stage 11)
Searches for the cheapest change in the levers the customer controls (card utilization, monthly debt, income) that brings the risk below the target. Age, late-payment history and dependents are reported as fixed factors only.

In [ ]:
from dataclasses import dataclass
from itertools import combinations, product
from typing import Callable, Sequence

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------
# Generic engine (dataset-agnostic: GMSC and UCI both plug into this)
# --------------------------------------------------------------------------
@dataclass
class Lever:
    """One thing the customer can change."""
    name: str
    apply: Callable[[pd.Series, float], pd.Series]   # (customer, amount) -> new customer
    grid: Callable[[pd.Series], Sequence[float]]     # candidate amounts, small -> large
    describe: Callable[[float], str]                 # English text for an amount
    describe_ar: Callable[[float], str]              # Arabic text for an amount
    weight: float = 1.0                              # effort multiplier (harder lever = bigger)


class RecommendationEngine:
    def __init__(self, risk_fn, levers, threshold, medium_cut=None,
                 explain_fn=None, fixed_drivers=(), level_fn=None, dataset=""):
        """
        risk_fn(list[pd.Series]) -> array of risks (batch, no SHAP, fast)
        explain_fn(customer)     -> list of {"driver", "effect", ...} (optional, for fixed factors)
        fixed_drivers            -> driver names the customer cannot change
        """
        self.risk_fn, self.levers = risk_fn, list(levers)
        self.threshold, self.medium_cut = threshold, medium_cut
        self.explain_fn, self.fixed_drivers = explain_fn, set(fixed_drivers)
        self.level_fn, self.dataset = level_fn, dataset

    # -- helpers ----------------------------------------------------------
    def _level(self, p):
        return self.level_fn(p) if self.level_fn else None

    def _plans(self, c, max_levers):
        grids = {}
        for lv in self.levers:
            try:
                g = [float(x) for x in lv.grid(c)]
            except ValueError:
                g = []
            if g:
                grids[lv.name] = g
        usable = [lv for lv in self.levers if lv.name in grids]
        for k in range(1, max_levers + 1):
            for combo in combinations(usable, k):
                for xs in product(*[grids[lv.name] for lv in combo]):
                    cost = sum(lv.weight * x / max(grids[lv.name]) for lv, x in zip(combo, xs))
                    yield list(zip(combo, xs)), cost

    def _evaluate(self, c, plans):
        """Apply each plan and score them all in one batch."""
        built, customers = [], []
        for steps, cost in plans:
            try:
                new = c.copy()
                for lv, x in steps:
                    new = lv.apply(new, x)
            except ValueError:
                continue
            built.append((steps, cost))
            customers.append(new)
        if not customers:
            return []
        risks = np.asarray(self.risk_fn(customers), dtype=float)
        return [(s, cst, float(r)) for (s, cst), r in zip(built, risks)]

    def _pack(self, steps, cost, risk, base_risk):
        return {
            "changes": [{"lever": lv.name, "amount": round(x, 4),
                         "text": lv.describe(x), "text_ar": lv.describe_ar(x)} for lv, x in steps],
            "risk_after": round(risk, 4),
            "delta": round(risk - base_risk, 4),
            "level_after": self._level(risk),
            "effort": round(cost, 3),
        }

    # -- main -------------------------------------------------------------
    def recommend(self, customer, target_risk=None, max_levers=2):
        """
        target_risk: risk must end up BELOW this value (default = model threshold,
                     i.e. "stop being flagged"; pass medium_cut to aim for Low).
        """
        c = pd.Series(customer) if not isinstance(customer, pd.Series) else customer.copy()
        target = self.threshold if target_risk is None else target_risk
        base = float(self.risk_fn([c])[0])

        out = {"dataset": self.dataset, "risk_now": round(base, 4), "level_now": self._level(base),
               "target_risk": round(target, 4), "fixed_factors": self._fixed_factors(c)}

        if base < target:
            out.update(status="already_below_target", recommendations=[],
                       message="Risk is already below the target; no change needed.",
                       message_ar="الخطر أصلاً أقل من الهدف، مفيش تغيير مطلوب.")
            return out

        for k in range(1, max_levers + 1):                 # fewest levers first
            evald = self._evaluate(c, [p for p in self._plans(c, k) if len(p[0]) == k])
            ok = [e for e in evald if e[2] < target]
            if ok:
                ok.sort(key=lambda e: e[1])                # cheapest first
                best = ok[0]
                # cheapest alternative per distinct lever-set (so the user has options)
                seen, alts = set(), []
                for steps, cost, risk in ok:
                    key = tuple(lv.name for lv, _ in steps)
                    if key not in seen:
                        seen.add(key)
                        alts.append(self._pack(steps, cost, risk, base))
                    if len(alts) == 3:
                        break
                out.update(status="found", best=alts[0], alternatives=alts[1:],
                           recommendations=alts,
                           message=" + ".join(ch["text"] for ch in alts[0]["changes"]),
                           message_ar=" + ".join(ch["text_ar"] for ch in alts[0]["changes"]))
                return out

        # nothing reaches the target: show the strongest achievable improvement
        evald = self._evaluate(c, self._plans(c, max_levers))
        steps, cost, risk = min(evald, key=lambda e: e[2]) if evald else (None, None, base)
        out.update(status="not_reachable", recommendations=[],
                   message="The target can't be reached with the levers the customer controls.",
                   message_ar="الهدف مش بيتحقق بالعوامل اللي العميل يقدر يغيّرها لوحدها.")
        if steps:
            out["best_effort"] = self._pack(steps, cost, risk, base)
        return out

    def _fixed_factors(self, c):
        if not self.explain_fn:
            return []
        return [r for r in self.explain_fn(c)
                if r["driver"] in self.fixed_drivers and r["effect"] == "increases risk"]


# --------------------------------------------------------------------------
# GMSC wiring
# --------------------------------------------------------------------------
def _gmsc_engine():
    import __main__ as g  # same notebook namespace

    def risk_batch(customers):
        df = pd.DataFrame([dict(c) for c in customers])
        X = g._prepare(g._as_frame(df))
        return g.CAL.predict(g.PIPE.predict_proba(X)[:, 1])

    # ---- levers ----
    def util_apply(c, pct):                       # pay the card down by pct of its balance
        c = c.copy()
        c["RevolvingUtilizationOfUnsecuredLines"] *= (1 - pct)
        return c

    def util_grid(c):
        u = c["RevolvingUtilizationOfUnsecuredLines"]
        if pd.isna(u) or u <= 0:
            return []
        return np.round(np.arange(0.05, 1.0001, 0.05), 2)

    def debt_apply(c, amount):                    # pay `amount`/month less in obligations
        g._need_income(c)
        c = c.copy()
        c["DebtRatio"] = max(0.0, c["DebtRatio"] - amount / c["MonthlyIncome"])
        return c

    def debt_grid(c):
        g._need_income(c)
        obligations = c["DebtRatio"] * c["MonthlyIncome"]
        if obligations <= 0:
            return []
        step = max(50, int(round(obligations / 20 / 50.0)) * 50)
        return list(np.arange(step, obligations + step, step))

    def income_apply(c, pct):                     # income rises by pct, obligations unchanged
        return g.income_drop(c, -pct)

    def income_grid(c):
        g._need_income(c)
        return np.round(np.arange(0.05, 0.5001, 0.05), 2)

    levers = [
        Lever("card_utilization", util_apply, util_grid,
              lambda x: f"Reduce credit-card utilization by {x:.0%}",
              lambda x: f"قلّل استخدام الكارت بنسبة {x:.0%}", weight=1.0),
        Lever("monthly_debt", debt_apply, debt_grid,
              lambda x: f"Pay down about {x:,.0f} of monthly debt obligations",
              lambda x: f"قلّل التزاماتك الشهرية بحوالي {x:,.0f}", weight=1.5),
        Lever("income", income_apply, income_grid,
              lambda x: f"Increase monthly income by {x:.0%}",
              lambda x: f"زوّد دخلك الشهري بنسبة {x:.0%}", weight=2.5),
    ]

    return RecommendationEngine(
        risk_fn=risk_batch, levers=levers, threshold=g.THRESHOLD, medium_cut=g.MEDIUM_CUT,
        explain_fn=lambda c: g.score_customer(c, top_k=6)["top_reasons"],
        fixed_drivers={"Late payments", "Age", "Dependents"},   # can't be changed by the user
        level_fn=g.risk_level, dataset="gmsc",
    )


_GMSC = None


def recommend_gmsc(customer, target_risk=None, max_levers=2):
    """customer: dict / Series with the raw GMSC columns (see build_raw_customer)."""
    global _GMSC
    if _GMSC is None:
        _GMSC = _gmsc_engine()
    import __main__ as g  # same notebook namespace
    c = g._as_frame(customer).iloc[0]
    return _GMSC.recommend(c, target_risk=target_risk, max_levers=max_levers)


# --------------------------------------------------------------------------
# UCI wiring (same engine, different levers)
# --------------------------------------------------------------------------
def recommend_uci(customer, target_risk=None, max_levers=2):
    raise NotImplementedError(
        "Needs the UCI scoring module (model, features, threshold). "
        "Once it exists: define risk_batch + levers (e.g. raise PAY_AMT_*, lower BILL_AMT_*, "
        "lower utilization of LIMIT_BAL) and build RecommendationEngine(...) like _gmsc_engine()."
    )


## 3. Savings Coverage (Stage 12, GMSC side)
**Rule-based, not model-based.** GMSC has no savings balance, so *liquid savings* is an input from the user (in their own currency, EGP). Nothing in this section touches the model, so it can't change the risk probabilities and it has no currency/training-distribution issue.

| Metric | Formula |
|---|---|
| `runway_months` | liquid_savings / (fixed_expenses + debt_payments) |
| `shock_coverage` | liquid_savings / shock_amount |
| `savings_after_shock` | liquid_savings - shock_amount |
| `months_to_depletion` | savings left / (monthly outflow - income), only when outflow > income |

`debt_payments = DebtRatio * MonthlyIncome` (same proxy used everywhere else in GMSC). Status bands (**critical / thin / safe**) come from `COVERAGE_CUTS` and are configurable. The overall status is the *worst* of the runway, shock and depletion statuses.

**How coverage sits next to the scenarios** (`run_all_with_coverage`):
- *Monthly* scenarios (income drop, new loan, higher bills): coverage is recomputed on the scenario customer (new obligations / new income), no one-off shock.
- *One-off* scenarios (medical emergency): the model sees the amount financed over 12 months (higher `DebtRatio`), while coverage sees the full amount taken out of savings. Coverage uses the **baseline** obligations here so the same shock isn't counted twice.

In [ ]:
# --------------------------------------------------------------------------
# Stage 12 - Savings Coverage (GMSC side): rule-based, NOT model-based
# --------------------------------------------------------------------------
COVERAGE_CUTS = {   # configurable; design decision: agree on these with the team
    "runway_months":  {"critical": 1.0, "thin": 3.0},   # < 1 month critical, < 3 thin, else safe
    "shock_coverage": {"critical": 1.0, "thin": 2.0},   # < 1x can't absorb the shock, < 2x thin, else safe
}
_STATUS_RANK = {"critical": 0, "thin": 1, "safe": 2}
_STATUS_AR = {"critical": "حرج", "thin": "ضعيف", "safe": "آمن"}


def _coverage_status(value, cut):
    return "critical" if value < cut["critical"] else "thin" if value < cut["thin"] else "safe"


def _worst(*statuses):
    return min(statuses, key=_STATUS_RANK.get)


def savings_coverage(customer, liquid_savings, fixed_expenses=0.0, shock_amount=0.0, cuts=None):
    """
    customer        : raw GMSC columns (dict / Series), see build_raw_customer
    liquid_savings  : cash the user can use right now (user input, EGP)
    fixed_expenses  : monthly living costs NOT already in DebtRatio (rent, food...). Default 0
    shock_amount    : one-off expense taken out of savings (e.g. medical emergency). Default 0
    """
    cuts = cuts or COVERAGE_CUTS
    if min(liquid_savings, fixed_expenses, shock_amount) < 0:
        raise ValueError("Savings, fixed expenses and shock amount can't be negative")
    c = _as_frame(customer).iloc[0]
    _need_income(c)

    income = float(c["MonthlyIncome"])
    debt_payments = float(c["DebtRatio"]) * income
    outflow = float(fixed_expenses) + debt_payments           # what leaves every month

    savings_after = liquid_savings - shock_amount
    usable_after = max(0.0, savings_after)
    gap = max(0.0, outflow - income)                          # monthly shortfall (0 if income covers it)

    runway = liquid_savings / outflow if outflow > 0 else None            # None = nothing to cover
    runway_after = usable_after / outflow if outflow > 0 else None
    shock_cov = liquid_savings / shock_amount if shock_amount > 0 else None
    depletion = usable_after / gap if gap > 0 else None                   # None = income covers outflows

    detail = {
        "runway": _coverage_status(runway_after, cuts["runway_months"]) if runway_after is not None else "safe",
        "shock": _coverage_status(shock_cov, cuts["shock_coverage"]) if shock_cov is not None else "safe",
        "depletion": _coverage_status(depletion, cuts["runway_months"]) if depletion is not None else "safe",
    }
    status = _worst(*detail.values())

    en = [f"Coverage is {status}."]
    ar = [f"التغطية {_STATUS_AR[status]}."]
    if runway_after is not None:
        en.append(f"Savings cover about {runway_after:.1f} months of fixed expenses + debt payments.")
        ar.append(f"مدخراتك بتغطي حوالي {runway_after:.1f} شهر من المصاريف الثابتة + أقساط الديون.")
    if shock_amount > 0:
        en.append(f"A {shock_amount:,.0f} shock leaves {savings_after:,.0f}" + (" (savings can't cover it)." if savings_after < 0 else "."))
        ar.append(f"صدمة بقيمة {shock_amount:,.0f} هتسيب {savings_after:,.0f}" + (" (المدخرات مش كفاية)." if savings_after < 0 else "."))
    if gap > 0:
        en.append(f"Monthly outflows exceed income by {gap:,.0f}; savings would last about {depletion:.1f} months.")
        ar.append(f"المصروفات الشهرية أكتر من الدخل بـ {gap:,.0f}؛ المدخرات هتكفي حوالي {depletion:.1f} شهر.")

    r2 = lambda v: None if v is None else round(float(v), 2)
    return {
        "dataset": "gmsc", "view": "savings_coverage", "method": "rule_based",
        "inputs": {"liquid_savings": liquid_savings, "fixed_expenses": fixed_expenses,
                   "shock_amount": shock_amount, "monthly_income": round(income, 2),
                   "debt_payments": round(debt_payments, 2)},
        "monthly_outflow": round(outflow, 2), "monthly_gap": round(gap, 2),
        "runway_months": r2(runway), "runway_after_shock_months": r2(runway_after),
        "shock_coverage": r2(shock_cov), "savings_after_shock": round(savings_after, 2),
        "months_to_depletion": r2(depletion),
        "status": status, "status_detail": detail,
        "message": " ".join(en), "message_ar": " ".join(ar),
    }


# --------------------------------------------------------------------------
# Coverage next to every scenario result
# --------------------------------------------------------------------------
ONE_OFF_SCENARIOS = {"Medical emergency (20,000 one-off)": 20000}   # name -> amount taken from savings
MEDICAL_FINANCING_MONTHS = 12   # model input mapping: one-off amount -> amount/12 extra monthly debt


def coverage_scenarios():
    """Stage-10 library + one-off shocks (built on call, so new SCENARIOS entries are picked up)."""
    lib = dict(SCENARIOS)
    for name, amount in ONE_OFF_SCENARIOS.items():
        lib[name] = lambda c, a=amount: add_monthly_payment(c, a / MEDICAL_FINANCING_MONTHS)
    return lib


def run_scenario_with_coverage(customer, name, liquid_savings, fixed_expenses=0.0):
    lib = coverage_scenarios()
    c = _as_frame(customer).iloc[0]
    res = run_scenario(c, name, change_fn=lib[name])           # model view (risk before/after)
    one_off = ONE_OFF_SCENARIOS.get(name, 0.0)
    after_customer = c if one_off else lib[name](c)             # one-off: baseline obligations, no double count
    res["coverage_before"] = savings_coverage(c, liquid_savings, fixed_expenses)
    res["coverage_after"] = savings_coverage(after_customer, liquid_savings, fixed_expenses, shock_amount=one_off)
    return res


def run_all_with_coverage(customer, liquid_savings, fixed_expenses=0.0):
    rows = []
    for name in coverage_scenarios():
        try:
            r = run_scenario_with_coverage(customer, name, liquid_savings, fixed_expenses)
            cb, ca = r["coverage_before"], r["coverage_after"]
            rows.append({"scenario": name,
                         "risk_before": r["before"], "risk_after": r["after"], "delta": r["delta"],
                         "level_before": r["level_before"], "level_after": r["level_after"],
                         "coverage_before": cb["status"], "coverage_after": ca["status"],
                         "runway_after_months": ca["runway_after_shock_months"],
                         "savings_left": ca["savings_after_shock"],
                         "months_to_depletion": ca["months_to_depletion"]})
        except ValueError as e:
            rows.append({"scenario": name, "error": str(e)})
    return pd.DataFrame(rows)

## 4. Long-Term Risk View, 24 months (Stage 13, GMSC)
GMSC is a **structural** view: the label is "serious delinquency (90+ days late) within 24 months" and the data has no time dimension, so it is **not** shown as 30/90-day risk and it is **never averaged or merged** with the UCI cards.

- Calibrated probability only, banded with the saved threshold (Low / Medium / High).
- All six driver groups (late payments, utilization, income, debt ratio, age, dependents) in plain language, in English and Arabic.
- Wording is "associated with higher/lower risk" (SHAP explains the model; it is not causal).
- Each driver is tagged *changeable* or *fixed*, matching the Recommendations Engine.
- The model predicts payment distress / delinquency (a proxy for a cash-flow crisis), **not** "running out of savings".

In [ ]:
# --------------------------------------------------------------------------
# Stage 13 - Long-term (24-month) structural risk view, GMSC
# --------------------------------------------------------------------------
FIXED_GROUPS = {"Late payments", "Age", "Dependents"}   # same split as the Recommendations Engine
_LATE_COLS = ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfTimes90DaysLate"]
_BAND = {
    "Low":    ("Low long-term structural risk",    "خطر هيكلي منخفض على المدى الطويل"),
    "Medium": ("Medium long-term structural risk", "خطر هيكلي متوسط على المدى الطويل"),
    "High":   ("High long-term structural risk",   "خطر هيكلي مرتفع على المدى الطويل"),
}


def _num(v, fmt):
    return "n/a" if pd.isna(v) else format(v, fmt)


def _driver_facts(c):
    """Plain-language description of the customer's actual value for each business group (EN, AR)."""
    late = int(sum(0 if pd.isna(c[k]) or c[k] in (96, 98) else c[k] for k in _LATE_COLS))
    income, dr, util = c["MonthlyIncome"], c["DebtRatio"], c["RevolvingUtilizationOfUnsecuredLines"]
    deps = 0 if pd.isna(c["NumberOfDependents"]) else int(c["NumberOfDependents"])
    pay = income * dr if pd.notna(income) and pd.notna(dr) else float("nan")
    return {
        "Late payments": (f"{late} late payment(s) of 30+ days on record" if late else "No late payments on record",
                          f"{late} تأخير في السداد (30 يوم أو أكتر) في السجل" if late else "مفيش تأخيرات في السداد في السجل"),
        "Credit utilization": (f"Credit cards are {_num(util, '.0%')} used",
                               f"استخدام الكروت {_num(util, '.0%')} من الحد"),
        "Income": (f"Monthly income {_num(income, ',.0f')}, with about {_num(pay, ',.0f')} going to debt payments",
                   f"الدخل الشهري {_num(income, ',.0f')}، وحوالي {_num(pay, ',.0f')} منه بيروح لأقساط الديون"),
        "Debt ratio": (f"Debt payments are {_num(dr, '.0%')} of income",
                       f"أقساط الديون بتمثل {_num(dr, '.0%')} من الدخل"),
        "Age": (f"Age {_num(c['age'], '.0f')}", f"السن {_num(c['age'], '.0f')}"),
        "Dependents": (f"{deps} dependent(s)", f"عدد المعالين {deps}"),
    }


def long_term_risk_view(customer):
    c = _as_frame(customer).iloc[0]
    s = score_customer(c, top_k=len(GROUPS))                 # all groups, ranked by |SHAP|
    facts = _driver_facts(c)
    drivers = []
    for r in s["top_reasons"]:
        higher = r["effect"] == "increases risk"
        en, ar = facts[r["driver"]]
        drivers.append({
            "group": r["driver"], "shap": r["shap"],
            "effect": "associated with higher risk" if higher else "associated with lower risk",
            "text": f"{en}: associated with {'higher' if higher else 'lower'} risk",
            "text_ar": f"{ar}: مرتبط بخطر {'أعلى' if higher else 'أقل'}",
            "changeable": r["driver"] not in FIXED_GROUPS,
        })
    head_en, head_ar = _BAND[s["level"]]
    return {
        "dataset": "gmsc", "view": "long_term_structural_risk", "horizon": "24 months",
        "risk": s["risk"], "level": s["level"], "flagged": s["flagged"],
        "threshold": s["threshold"], "medium_cut": round(MEDIUM_CUT, 4),
        "headline": f"{head_en}: {s['risk']:.1%} estimated chance of serious payment delinquency (90+ days late) within 24 months",
        "headline_ar": f"{head_ar}: احتمال {s['risk']:.1%} لتأخر سداد خطير (90 يوم أو أكتر) خلال 24 شهر",
        "drivers": drivers,
        "note": ("Structural 24-month view of payment distress / delinquency. This dataset has no time dimension, "
                 "so no 30/90-day split is shown. Drivers explain the model; they are associations, not causes. "
                 "It does not measure running out of savings."),
        "note_ar": ("ده عرض هيكلي لخطر التعثر في السداد على 24 شهر. البيانات مفيهاش بُعد زمني فمفيش تقسيم 30/90 يوم. "
                    "العوامل بتشرح النموذج وهي ارتباطات مش أسباب، ومش بتقيس نفاد المدخرات."),
    }


def print_long_term_view(v, lang="en"):
    ar = lang == "ar"
    print(v["headline_ar" if ar else "headline"])
    for d in v["drivers"]:
        print(f"  [{'قابل للتغيير' if ar and d['changeable'] else 'ثابت' if ar else 'changeable' if d['changeable'] else 'fixed'}] "
              f"{d['text_ar' if ar else 'text']}  (SHAP {d['shap']:+.3f})")
    print(v["note_ar" if ar else "note"])


def plot_long_term_view(v):
    import matplotlib.pyplot as plt
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1.3]})
    top = max(v["threshold"] * 1.6, v["risk"] * 1.15)
    a1.barh([0], [v["risk"]], color="#d62728" if v["flagged"] else "#ff7f0e" if v["level"] == "Medium" else "#2ca02c")
    for x, lbl in [(v["medium_cut"], "Medium"), (v["threshold"], "High")]:
        a1.axvline(x, color="k", ls="--", lw=1); a1.text(x, 0.45, lbl, ha="center", fontsize=8)
    a1.set_xlim(0, top); a1.set_yticks([]); a1.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    a1.set_title(f"24-month structural risk: {v['risk']:.1%} ({v['level']})")
    ds = sorted(v["drivers"], key=lambda d: abs(d["shap"]))
    a2.barh([d["group"] + ("" if d["changeable"] else " (fixed)") for d in ds], [d["shap"] for d in ds],
            color=["#d62728" if d["shap"] > 0 else "#2ca02c" for d in ds])
    a2.axvline(0, color="k", lw=0.8)
    a2.set_title("Drivers: red = associated with higher risk, green = lower")
    plt.tight_layout(); plt.show()

## 5. Demo

In [ ]:
import json

demo = build_raw_customer(age=35, monthly_income=8000, monthly_debt_payments=5200,
                          card_balance=9000, card_limit=10000, late_30_59=1,
                          dependents=2, open_credit_lines=6)

print("=== Score ===")
print(json.dumps(score_customer(demo), indent=2, ensure_ascii=False))

print("\n=== What-if scenarios (Stage 10) ===")
display(run_all(demo))

print("\n=== Recommendations: get out of High ===")
print(json.dumps(recommend_gmsc(demo), indent=2, ensure_ascii=False))

print("\n=== Recommendations: get down to Low ===")
print(json.dumps(recommend_gmsc(demo, target_risk=MEDIUM_CUT), indent=2, ensure_ascii=False))


In [ ]:
# --------------------------------------------------------------------------
# Stage 12 + 13 demo and sanity checks
# --------------------------------------------------------------------------
SAVINGS = 15000   # EGP, user input

print("=== Savings coverage (baseline) ===")
print(json.dumps(savings_coverage(demo, SAVINGS, fixed_expenses=2000), indent=2, ensure_ascii=False))

print("\n=== Scenarios with coverage next to them ===")
display(run_all_with_coverage(demo, SAVINGS, fixed_expenses=2000))

print("\n=== Long-term (24-month) risk view ===")
view = long_term_risk_view(demo)
print(json.dumps(view, indent=2, ensure_ascii=False))
print_long_term_view(view)
print_long_term_view(view, lang="ar")
plot_long_term_view(view)

# ---- sanity checks (no retraining, no test-set use) ----
# coverage: more savings never makes things worse; bigger shock never makes things better
lo, hi = savings_coverage(demo, 5000, 2000), savings_coverage(demo, 50000, 2000)
assert hi["runway_months"] > lo["runway_months"]
assert _STATUS_RANK[hi["status"]] >= _STATUS_RANK[lo["status"]]
s_small, s_big = savings_coverage(demo, SAVINGS, 2000, 1000), savings_coverage(demo, SAVINGS, 2000, 9000)
assert s_big["savings_after_shock"] < s_small["savings_after_shock"]
assert _STATUS_RANK[s_big["status"]] <= _STATUS_RANK[s_small["status"]]
assert savings_coverage(demo, 0, 2000)["status"] == "critical"
# coverage must not change the model's output
assert view["risk"] == score_customer(demo)["risk"]
# long-term view: 24 months, no 30/90-day framing, associative wording only
assert view["horizon"] == "24 months" and not any(k in view for k in ("risk_30", "risk_90"))
assert all("cause" not in d["text"].lower() for d in view["drivers"])
assert {d["group"] for d in view["drivers"]} == set(GROUPS)
print("\nAll Stage 12/13 checks passed.")